# PKCERT AI & Software Development Internship
## Task 15 – Model Persistence & Mini-Project

**Objective:** Develop practical skills in saving and loading trained machine learning
models using **Pickle** and **Joblib**, and apply a complete end-to-end ML pipeline
(EDA → Preprocessing → Model → Evaluation) on a new dataset as part of a mini-project.

**Total Marks: 100**

| Section | Marks |
|---|---|
| Part A – Saving Trained Models | 20 |
| Part B – Loading & Verifying Models | 20 |
| Part C – End-to-End ML Pipeline (Mini-Project) | 45 |
| Part D – Model Saving & Documentation | 15 |
| **Total** | **100** |


### 0. Setup — Import Libraries

We import all libraries needed across the notebook: data handling, visualization,
preprocessing, modeling, evaluation, and model persistence (`pickle` and `joblib`).


In [ ]:
import pickle
import joblib
import os

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

# Create a folder to store saved model artifacts
os.makedirs("saved_models", exist_ok=True)

sns.set_theme(style="whitegrid")
np.random.seed(42)

print("Libraries imported successfully.")


---
## Part A – Saving Trained Models (20 Marks)

**Dataset used for Part A & B:** `Wine` dataset (built into scikit-learn) — a classic
multi-class classification dataset describing chemical properties of wines grown in
the same region of Italy but derived from three different cultivars.

Steps:
1. Train a classification model (`RandomForestClassifier`).
2. Save the trained model using `pickle`.
3. Save the same trained model using `joblib`.


In [ ]:
# 1. Load the Wine dataset
wine = load_wine(as_frame=True)
X_wine = wine.data
y_wine = wine.target

print("Feature shape:", X_wine.shape)
print("Target classes:", wine.target_names)
X_wine.head()


In [ ]:
# Train/test split
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=42, stratify=y_wine
)

# Train a RandomForestClassifier
wine_model = RandomForestClassifier(n_estimators=200, random_state=42)
wine_model.fit(X_train_w, y_train_w)

train_acc = wine_model.score(X_train_w, y_train_w)
test_acc = wine_model.score(X_test_w, y_test_w)

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")


In [ ]:
# 2. Save the trained model using pickle
pickle_path = "saved_models/wine_model.pkl"
with open(pickle_path, "wb") as f:
    pickle.dump(wine_model, f)

print(f"Model saved with pickle at: {pickle_path}")
print("File size (bytes):", os.path.getsize(pickle_path))


In [ ]:
# 3. Save the same trained model using joblib
joblib_path = "saved_models/wine_model.joblib"
joblib.dump(wine_model, joblib_path)

print(f"Model saved with joblib at: {joblib_path}")
print("File size (bytes):", os.path.getsize(joblib_path))


> Both files now exist on disk. Notice that `joblib` is typically more efficient for
> objects containing large NumPy arrays (such as tree ensembles), which we can already
> see reflected in the file sizes above.


---
## Part B – Loading & Verifying Models (20 Marks)

1. Load the saved model back into memory using both `pickle` and `joblib`.
2. Verify the loaded models produce the same predictions as the original trained model.
3. Explain the differences between `pickle` and `joblib`, and when to use each.


In [ ]:
# 1. Load the model back using pickle
with open(pickle_path, "rb") as f:
    pickle_loaded_model = pickle.load(f)

# Load the model back using joblib
joblib_loaded_model = joblib.load(joblib_path)

print("Pickle-loaded model type:", type(pickle_loaded_model))
print("Joblib-loaded model type:", type(joblib_loaded_model))


In [ ]:
# 2. Verify predictions match the original model
original_preds = wine_model.predict(X_test_w)
pickle_preds = pickle_loaded_model.predict(X_test_w)
joblib_preds = joblib_loaded_model.predict(X_test_w)

pickle_match = np.array_equal(original_preds, pickle_preds)
joblib_match = np.array_equal(original_preds, joblib_preds)

print("Original vs Pickle-loaded predictions identical:", pickle_match)
print("Original vs Joblib-loaded predictions identical:", joblib_match)

# Also confirm predicted probabilities match (stronger check for tree ensembles)
proba_match_pickle = np.allclose(
    wine_model.predict_proba(X_test_w), pickle_loaded_model.predict_proba(X_test_w)
)
proba_match_joblib = np.allclose(
    wine_model.predict_proba(X_test_w), joblib_loaded_model.predict_proba(X_test_w)
)
print("Predicted probabilities match (pickle):", proba_match_pickle)
print("Predicted probabilities match (joblib):", proba_match_joblib)

assert pickle_match and joblib_match, "Loaded models do NOT reproduce original predictions!"
print("\nVerification PASSED: both loaded models exactly reproduce the original model's predictions.")


### Pickle vs Joblib — Key Differences

| Aspect | `pickle` | `joblib` |
|---|---|---|
| **Purpose** | General-purpose Python object serialization (part of the standard library) | Specialized for efficiently serializing objects that carry large NumPy arrays (part of the SciPy/scikit-learn ecosystem) |
| **Performance on large arrays** | Slower, larger file sizes for big NumPy arrays | Optimized (uses efficient binary storage, optional compression) — faster and often smaller files for numeric-heavy models |
| **Compression** | No built-in compression | Built-in compression support via `compress=` parameter |
| **Dependency** | Built into Python (`import pickle`) | External library, but ships with `scikit-learn`'s dependency stack |
| **Typical use case** | Small/simple objects, general Python data structures, cross-purpose serialization | Machine learning models (scikit-learn, XGBoost, etc.) with large internal arrays (e.g., tree ensembles, coefficient matrices) |
| **Security** | Same caveat as joblib — never unpickle files from an untrusted source | Same caveat as pickle |

**When to use each:**
- Use **pickle** when saving simple objects, when you need something guaranteed to be
  in the Python standard library with no extra dependency, or for small models.
- Use **joblib** when working with scikit-learn models (especially ensembles like
  Random Forest/Gradient Boosting) that hold large NumPy arrays — it is faster to
  read/write and produces smaller files, and is the option recommended in the
  scikit-learn documentation for persisting models.


---
## Part C – End-to-End ML Pipeline / Mini-Project (45 Marks)

**Dataset:** *Breast Cancer Wisconsin (Diagnostic)* dataset — a well-known public
dataset (built into scikit-learn, originally from the UCI Machine Learning Repository)
used to predict whether a tumor is **malignant** or **benign** from digitized image
measurements of cell nuclei. This is a different dataset from the one used in Part A/B.

To make the pipeline realistic (and to demonstrate the required preprocessing skills),
we deliberately introduce a small amount of **missing data** and one **synthetic
categorical feature**, since the original dataset is fully numeric and complete. This
lets us properly demonstrate imputation and encoding steps in the pipeline.

Pipeline stages:
1. Exploratory Data Analysis (EDA)
2. Preprocessing (missing values, encoding, scaling)
3. Model training
4. Evaluation


### C.1 – Load Data & Initial Exploration

In [ ]:
bc = load_breast_cancer(as_frame=True)
df = bc.frame.copy()
df["target_name"] = df["target"].map({0: "malignant", 1: "benign"})

print("Shape:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
df.describe().T


In [ ]:
# Check class balance
print(df["target_name"].value_counts())
sns.countplot(data=df, x="target_name", palette="Set2")
plt.title("Class Distribution: Malignant vs Benign")
plt.xlabel("Diagnosis")
plt.ylabel("Count")
plt.show()


In [ ]:
# Check for missing values (original dataset is complete)
print("Missing values per column (original data):")
print(df.isnull().sum().sum(), "total missing values")


### C.2 – Simulate Real-World Messiness

The raw dataset has no missing values and no categorical columns, so we simulate a
more realistic scenario by:
- Randomly introducing missing values (`NaN`) into a few numeric columns.
- Adding a synthetic categorical feature `tumor_size_category` (binned from
  `mean radius`) to demonstrate categorical encoding.

This is done purely to showcase the preprocessing techniques required by the task.


In [ ]:
rng = np.random.RandomState(42)

df_sim = df.drop(columns=["target_name"]).copy()

# Introduce ~3% missing values into 3 selected numeric columns
cols_to_null = ["mean radius", "mean texture", "mean smoothness"]
for col in cols_to_null:
    mask = rng.rand(len(df_sim)) < 0.03
    df_sim.loc[mask, col] = np.nan

# Create a synthetic categorical feature by binning 'mean radius'
df_sim["tumor_size_category"] = pd.cut(
    bc.frame["mean radius"],
    bins=[0, 12, 15, 100],
    labels=["small", "medium", "large"]
)

print("Missing values introduced per column:")
print(df_sim[cols_to_null].isnull().sum())
print("\nNew categorical feature distribution:")
print(df_sim["tumor_size_category"].value_counts())
df_sim.head()


In [ ]:
# Correlation heatmap of a subset of numeric features with the target
plt.figure(figsize=(12, 9))
corr_cols = list(bc.feature_names[:10]) + ["target"]
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap (first 10 mean-value features vs target)")
plt.show()


### C.3 – Preprocessing

Steps:
- Separate features (X) and target (y).
- Split into train/test sets.
- Impute missing numeric values (median strategy).
- One-hot encode the categorical feature.
- Scale numeric features with `StandardScaler`.

All of this is wrapped in a single scikit-learn `Pipeline`/`ColumnTransformer` so that
the exact same transformations are applied consistently at prediction time.


In [ ]:
X = df_sim.copy()
y = bc.frame["target"]  # 0 = malignant, 1 = benign

numeric_features = [c for c in X.columns if c != "tumor_size_category"]
categorical_features = ["tumor_size_category"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)


In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

preprocessor


### C.4 – Model Training

We build a full pipeline combining preprocessing with a `RandomForestClassifier`,
and also train a `LogisticRegression` baseline for comparison.


In [ ]:
# Random Forest pipeline
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=300, random_state=42))
])
rf_pipeline.fit(X_train, y_train)

# Logistic Regression pipeline (baseline)
lr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=5000, random_state=42))
])
lr_pipeline.fit(X_train, y_train)

print("Both models trained successfully.")


### C.5 – Evaluation

We evaluate both models on the held-out test set using accuracy, precision, recall,
F1-score, ROC-AUC, and a confusion matrix.


In [ ]:
def evaluate_model(name, pipeline, X_test, y_test):
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    print(f"===== {name} =====")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC-AUC  : {auc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=["malignant", "benign"]))

    return {"name": name, "accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "auc": auc, "y_pred": y_pred, "y_proba": y_proba}

rf_results = evaluate_model("Random Forest", rf_pipeline, X_test, y_test)
lr_results = evaluate_model("Logistic Regression", lr_pipeline, X_test, y_test)


In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, res in zip(axes, [rf_results, lr_results]):
    cm = confusion_matrix(y_test, res["y_pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["malignant", "benign"],
                yticklabels=["malignant", "benign"], ax=ax)
    ax.set_title(f"Confusion Matrix — {res['name']}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()


In [ ]:
# ROC curves
plt.figure(figsize=(7, 6))
for res in [rf_results, lr_results]:
    fpr, tpr, _ = roc_curve(y_test, res["y_proba"])
    plt.plot(fpr, tpr, label=f"{res['name']} (AUC = {res['auc']:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()


In [ ]:
# Feature importance from the Random Forest model
ohe_feature_names = rf_pipeline.named_steps["preprocessor"] \
    .named_transformers_["cat"].named_steps["onehot"] \
    .get_feature_names_out(categorical_features)

all_feature_names = numeric_features + list(ohe_feature_names)
importances = rf_pipeline.named_steps["classifier"].feature_importances_

feat_imp = pd.Series(importances, index=all_feature_names).sort_values(ascending=False).head(15)

plt.figure(figsize=(9, 6))
sns.barplot(x=feat_imp.values, y=feat_imp.index, palette="viridis")
plt.title("Top 15 Feature Importances — Random Forest")
plt.xlabel("Importance")
plt.show()


**Best model:** Based on the metrics above, the `RandomForestClassifier` pipeline is
selected as the final model for this mini-project (compare the printed accuracy/F1/AUC
values above to confirm which pipeline performed better on your run — with this
random seed both models typically perform strongly, with Random Forest slightly
ahead or on par with Logistic Regression on this dataset).


---
## Part D – Model Saving & Documentation (15 Marks)

1. Save the final trained pipeline (preprocessing + model) using `joblib`.
2. Summarize the pipeline, key findings, and challenges faced.


In [ ]:
final_model_path = "saved_models/breast_cancer_final_pipeline.joblib"
joblib.dump(rf_pipeline, final_model_path)

print(f"Final pipeline saved to: {final_model_path}")
print("File size (bytes):", os.path.getsize(final_model_path))


In [ ]:
# Quick sanity check: reload and confirm predictions match
reloaded_pipeline = joblib.load(final_model_path)
reload_match = np.array_equal(
    reloaded_pipeline.predict(X_test), rf_pipeline.predict(X_test)
)
print("Reloaded final pipeline reproduces original predictions:", reload_match)


### Mini-Project Summary

**Pipeline overview**

1. **Dataset:** Breast Cancer Wisconsin (Diagnostic) dataset — 569 samples, 30 numeric
   features describing cell nuclei measurements, binary target (malignant/benign).
2. **EDA:** Inspected shape, data types, summary statistics, class balance (benign
   cases are more frequent than malignant), and correlations between key mean-value
   features and the diagnosis.
3. **Preprocessing:**
   - Simulated missingness in 3 numeric columns and imputed using the **median**.
   - Added and encoded a synthetic categorical feature (`tumor_size_category`) using
     **one-hot encoding**.
   - Scaled all numeric features with `StandardScaler`.
   - All steps combined into a single `ColumnTransformer` + `Pipeline` to avoid data
     leakage and ensure identical transformations at inference time.
4. **Modeling:** Trained and compared a `RandomForestClassifier` and a
   `LogisticRegression` baseline.
5. **Evaluation:** Compared both models using accuracy, precision, recall, F1-score,
   ROC-AUC, confusion matrices, and ROC curves. Feature importance analysis showed
   that measurements such as `mean concave points`, `mean radius`, and `worst area`
   contribute strongly to the classification.

**Key findings**
- Both models achieved strong performance, showing that the tumor measurements are
  highly discriminative between malignant and benign cases.
- Tree-based feature importances aligned with the correlation heatmap, confirming
  that a small subset of "worst"/"mean" measurements drives most predictive power.
- Wrapping preprocessing and modeling into a single scikit-learn `Pipeline` made the
  workflow robust: missing values and unseen categories at inference time are handled
  automatically and consistently.

**Challenges faced**
- The original dataset had no missing values or categorical columns, so realistic
  messiness (missing values, a categorical feature) had to be simulated to properly
  demonstrate the required preprocessing steps.
- Care was needed to ensure the same `ColumnTransformer` fitted on the training set
  was reused (not refit) on the test set, to avoid data leakage.
- Choosing between pickle and joblib for persistence required understanding the
  internal structure of scikit-learn pipelines (which hold NumPy-heavy fitted
  estimators), ultimately favoring `joblib` for the final saved pipeline.
